# Überprüfung auf Length Bias (Längen-Shortcut-Lernen)

In diesem Notebook überprüfen wir empirisch, ob unser bestes BiLSTM-Modell (`lstm_article_sim_0.80_to_0.98.pt`) gelernt hat, die Artikelklasse (Leichte Sprache [LS] vs. Alltagssprache [AS]) primär über die Länge des Textes (oder das Vorhandensein von Padding-Nullen) zu bestimmen.

Dazu führen wir drei Experimente durch:
1. **Korrelationsanalyse**: Gibt es einen statistisch signifikanten Zusammenhang zwischen Textlänge und Vorhersagewahrscheinlichkeit?
2. **Dummy-Text-Experiment (Konstanter Token-Test)**: Wir ersetzen alle Wörter durch einen Punkt `.`, behalten aber die Originallängen bei. Kann das Modell rein über die Länge/Padding-Struktur klassifizieren?
3. **Festlängen-Evaluation**: Wir schneiden alle Texte auf eine feste Länge (z. B. 100 Token) ab, sodass alle Eingaben identisch lang sind. Bleibt die Klassifikationsgenauigkeit hoch?

In [ ]:
import os
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import pearsonr, spearmanr
from sklearn.metrics import balanced_accuracy_score, classification_report
from IPython.display import display

def find_repo_root():
    p = os.path.abspath(os.getcwd())
    while p != os.path.dirname(p):
        if os.path.exists(os.path.join(p, "data")) and os.path.exists(os.path.join(p, "results")):
            return p
        p = os.path.dirname(p)
    return os.path.abspath(os.path.expanduser("~/Documents/Master Thesis"))

REPO_ROOT = find_repo_root()
os.chdir(REPO_ROOT)
if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)

print("Arbeitsverzeichnis:", os.getcwd())

sns.set_theme(style="whitegrid", palette="deep")
plt.rcParams.update({
    "font.family": "sans-serif",
    "font.size": 11,
    "axes.labelsize": 12,
    "axes.titlesize": 13,
    "figure.dpi": 150
})


## 1. Length-Bias Evaluationsergebnisse laden

In [ ]:
CSV_PATH = os.path.join(REPO_ROOT, "results/evaluation/length_bias_results.csv")

if not os.path.exists(CSV_PATH):
    print(f"[HINWEIS] {CSV_PATH} nicht gefunden.")
    print("Erzeuge Fallback-Datensatz zur Demonstration. Führe für echte Messungen sbatch scripts/sbatch/experiments/metric/length_bias/1_check_length_bias.sh aus.")
    np.random.seed(42)
    as_lens = np.random.normal(380, 90, 37).clip(150, 700).astype(int)
    ls_lens = np.random.normal(210, 60, 37).clip(80, 450).astype(int)
    all_lens = np.concatenate([as_lens, ls_lens])
    labels = np.array([0]*37 + [1]*37)
    probs = np.concatenate([np.random.uniform(0.02, 0.25, 37), np.random.uniform(0.75, 0.98, 37)])
    dummy_p = np.array([1]*74)
    f50_p = np.concatenate([np.random.choice([0, 1], 37, p=[0.7, 0.3]), np.random.choice([0, 1], 37, p=[0.3, 0.7])])
    f100_p = np.concatenate([np.random.choice([0, 1], 37, p=[0.88, 0.12]), np.random.choice([0, 1], 37, p=[0.12, 0.88])])
    df_length_bias = pd.DataFrame({
        "length": all_lens,
        "probability": probs,
        "true_label": labels,
        "dummy_pred": dummy_p,
        "fixed_50_pred": f50_p,
        "fixed_100_pred": f100_p
    })
else:
    df_length_bias = pd.read_csv(CSV_PATH)
    print(f"Ergebnisse erfolgreich geladen: {len(df_length_bias)} Zeilen aus {CSV_PATH}")

lengths = df_length_bias["length"].values
probabilities = df_length_bias["probability"].values
true_labels = df_length_bias["true_label"].values
dummy_preds = df_length_bias["dummy_pred"].values
fixed_len_preds_50 = df_length_bias["fixed_50_pred"].values
fixed_len_preds_100 = df_length_bias["fixed_100_pred"].values
preds = (probabilities > 0.5).astype(int)

display(df_length_bias.head())


## 2. Experiment 1: Korrelationsanalyse & Scatter Plot

Wir analysieren den linearen (Pearson) und monotonen (Spearman) Zusammenhang zwischen der Länge eines Textes und der Wahrscheinlichkeit, mit der das Modell den Text als "Simple" (LS) einstuft.

In [ ]:
pearson_r, p_val_p = pearsonr(lengths, probabilities)
spearman_r, p_val_s = spearmanr(lengths, probabilities)

print(f"Pearson Korrelation r:  {pearson_r:.4f} (p-Wert: {p_val_p:.4e})")
print(f"Spearman Korrelation r: {spearman_r:.4f} (p-Wert: {p_val_s:.4e})")

# Scatter Plot visualisieren
df_plot = pd.DataFrame({
    "Textlaenge (Worte)": lengths,
    "Wahrscheinlichkeit fuer Leichte Sprache (LS)": probabilities,
    "Klasse": ["LS (Simple)" if l == 1 else "AS (Normal)" for l in true_labels]
})

plt.figure(figsize=(10, 6))
sns.scatterplot(
    data=df_plot,
    x="Textlaenge (Worte)",
    y="Wahrscheinlichkeit fuer Leichte Sprache (LS)",
    hue="Klasse",
    palette={"LS (Simple)": "#2ecc71", "AS (Normal)": "#e74c3c"},
    alpha=0.8,
    s=80
)
plt.axhline(0.5, color="grey", linestyle="--", label="Entscheidungsgrenze")
plt.title(f"Zusammenhang Textlaenge vs. Modellkonfidenz\n(Pearson r = {pearson_r:.3f}, Spearman r = {spearman_r:.3f})")
plt.ylim(-0.05, 1.05)
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
plt.tight_layout()
plt.show()

## 3. Experiment 2: Constant Token / Dummy-Text-Test

Indem wir jedes Wort durch ein neutrales Dummy-Zeichen (z. B. `.`) ersetzen, löschen wir jeglichen Inhalt. Nur die Originallänge bleibt erhalten.
Kann das Modell diese inhaltsleeren Texte immer noch richtig zuordnen?

In [ ]:
dummy_bacc = balanced_accuracy_score(true_labels, dummy_preds)
print(f"Balanced Accuracy auf Dummy-Texten: {dummy_bacc*100:.2f}%\n")
print("Klassifikationsbericht für Dummy-Texte:")
print(classification_report(true_labels, dummy_preds, target_names=["AS (Normal)", "LS (Simple)"], zero_division=0))

## 4. Experiment 3: Festlängen-Evaluation

Wir schneiden alle Texte auf exakt die ersten $N$ Wörter ab ($N = 50$ und $N = 100$). Dadurch haben alle Texte im Datensatz exakt die gleiche Länge und das identische Padding-Muster. Wenn das Modell hier weiterhin gut performt, liegt das an den Wort- und Satzstrukturen innerhalb dieser Wortfenster.

In [ ]:
bacc_original = balanced_accuracy_score(true_labels, preds)
bacc_50 = balanced_accuracy_score(true_labels, fixed_len_preds_50)
bacc_100 = balanced_accuracy_score(true_labels, fixed_len_preds_100)

print(f"Balanced Accuracy (Volle Sequenzen, max 512 Token):  {bacc_original*100:.2f}%")
print(f"Balanced Accuracy (Limitiert auf exakt 100 Token): {bacc_100*100:.2f}%")
print(f"Balanced Accuracy (Limitiert auf exakt 50 Token):  {bacc_50*100:.2f}%")

# Balkendiagramm der Genauigkeiten
acc_df = pd.DataFrame({
    "Experiment Setup": [
        "Full Articles (max 512 Token)",
        "100 Token",
        "50 Token",
        "Dummy texts"
    ],
    "Balanced Accuracy": [
        bacc_original, 
        bacc_100, 
        bacc_50, 
        dummy_bacc
    ]
})

plt.figure(figsize=(10, 6))
ax = sns.barplot(
    data=acc_df,
    x="Experiment Setup",
    y="Balanced Accuracy",
    palette="viridis"
)
plt.ylim(0, 1.05)
plt.title("Model Accuracy Comparison Across Different Length Constraints")
plt.ylabel("Balanced Accuracy")

for p in ax.patches:
    ax.annotate(
        f"{p.get_height()*100:.2f}%", 
        (p.get_x() + p.get_width() / 2., p.get_height() + 0.01), 
        ha='center', 
        va='center', 
        xytext=(0, 8), 
        textcoords='offset points',
        fontweight='bold'
    )

plt.tight_layout()
plt.show()

## 5. Zusatz: Verteilung der Textlängen im Datensatz

Um zu verstehen, ob das Modell überhaupt einen Unterschied in den Längenverteilungen zum Lernen zur Verfügung gehabt hätte, plotten wir die Wortanzahl pro Klasse.

In [ ]:
df_len = pd.DataFrame({
    "Wortanzahl": lengths,
    "Klasse": ["LS (Simple)" if l == 1 else "AS (Normal)" for l in true_labels]
})

plt.figure(figsize=(10, 6))
sns.histplot(
    data=df_len,
    x="Wortanzahl",
    hue="Klasse",
    kde=True,
    element="step",
    stat="density",
    common_norm=False,
    palette={"LS (Simple)": "#2ecc71", "AS (Normal)": "#e74c3c"},
    alpha=0.5
)
plt.title("Verteilung der Wortanzahl nach Textklasse im Lebenshilfe-Set")
plt.xlabel("Tatsaechliche Wortanzahl (Tokens)")
plt.ylabel("Dichte")
plt.tight_layout()
plt.show()